# N27 — Race Situation Agent

This notebook builds the **Race Situation Agent**, the third sub-agent in the F1
multi-agent system. It answers two questions per lap:

- **Can we overtake?** — probability that driver X passes the car directly ahead
  within the next few laps, given the current gap, pace delta, and tyre context.
- **Is a Safety Car coming?** — probability of a SC deployment within the next
  3 laps, given the current race chaos level.

Both questions are answered by LightGBM classifiers trained in N12 and N14,
wrapped in LangChain `@tool` functions and orchestrated by a LangGraph ReAct agent.

Models loaded from `data/models/`:

| File | Source | Contents |
|---|---|---|
| `overtake_probability/lgbm_overtake_v1.pkl` | N12 | LightGBM overtake classifier |
| `overtake_probability/calibrator.pkl` | N12 | Platt calibrator (val 2024) |
| `overtake_probability/model_config.json` | N12 | Features, threshold, CAT_FEATURES |
| `safety_car_probability/lgbm_sc_v1.pkl` | N14 | LightGBM SC classifier |
| `safety_car_probability/calibrator_sc_v1.pkl` | N14 | Platt calibrator |
| `safety_car_probability/feature_list_v1.json` | N14 | Feature list (ordered) |

Output consumed by N31 Orchestrator to condition the pit strategy decision in N28.


---

## Step 0 — Setup & model loading

Imports, repo root, FastF1 cache. `RaceSituationConfig` loads both model pairs
(overtake + SC) and the circuit cluster map. All models serialised with `joblib`
(required on Windows paths containing non-ASCII characters).


In [1]:
import json, sys
from dataclasses import dataclass, field
from pathlib import Path

import fastf1
import joblib
import numpy as np
import pandas as pd

repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent


c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@dataclass
class RaceSituationConfig:
    """Runtime configuration for the Race Situation Agent.

    Loads both LightGBM model pairs (overtake from N12, SC from N14) plus their
    Platt calibrators and feature lists. Models are loaded with joblib because the
    repo path contains non-ASCII characters that break LightGBM's native save_model
    on Windows.

    overtake_threshold and sc_threshold are the classification thresholds derived
    from the Optuna/F2-score tuning in N12 and N14 respectively. threat_level
    boundaries (high_overtake, high_sc, medium_overtake, medium_sc) are agent-level
    decision rules that map raw probabilities to LOW/MEDIUM/HIGH for N31.
    """

    model_name: str = "local-model"

    # Threat-level boundaries
    high_overtake: float   = 0.80   # N12 optimal threshold — above this → HIGH
    medium_overtake: float = 0.40
    high_sc: float         = 0.30   # above this → HIGH regardless of overtake
    medium_sc: float       = 0.15

    def __post_init__(self):
        root = Path.cwd()
        while not (root / ".git").exists():
            root = root.parent

        fastf1.Cache.enable_cache(str(root / "data" / "cache" / "fastf1"))

        self.export_dir = root / "data" / "models" / "agents"
        self.export_dir.mkdir(parents=True, exist_ok=True)

        # --- Overtake model (N12) ---
        _ov_dir = root / "data" / "models" / "overtake_probability"
        self.overtake_model      = joblib.load(_ov_dir / "lgbm_overtake_v1.pkl")
        self.overtake_calibrator = joblib.load(_ov_dir / "calibrator.pkl")
        with open(_ov_dir / "model_config.json") as f:
            ov_cfg = json.load(f)
        self.overtake_features: list[str]     = ov_cfg["features"]
        self.overtake_cat_features: list[str] = ov_cfg["categorical_features"]
        self.overtake_threshold: float        = ov_cfg["optimal_threshold"]

        # --- SC model (N14) ---
        _sc_dir = root / "data" / "models" / "safety_car_probability"
        self.sc_model      = joblib.load(_sc_dir / "lgbm_sc_v1.pkl")
        self.sc_calibrator = joblib.load(_sc_dir / "calibrator_sc_v1.pkl")
        with open(_sc_dir / "feature_list_v1.json") as f:
            _sc_cfg = json.load(f)
        self.sc_features: list[str] = _sc_cfg["features"]
        self.sc_threshold: float    = _sc_cfg["best_threshold"]

        # --- Circuit cluster map (N03) — keyed by GP_Name ---
        _cluster_df = pd.read_parquet(
            root / "data" / "processed" / "circuit_clustering" / "circuit_clusters_k4.parquet"
        )
        self.circuit_cluster_map: dict = dict(
            zip(_cluster_df["GP_Name"], _cluster_df["Cluster"].astype(int))
        )

        # --- Circuit SC base rates (N13) — keyed by event_name ---
        _sc_rates_path = root / "data" / "processed" / "sc_labeled" / "sc_labeled_2023_2025.parquet"
        _sc_df = pd.read_parquet(_sc_rates_path, columns=["event_name", "circuit_sc_rate"])
        self.circuit_sc_rate_map: dict = (
            _sc_df.drop_duplicates("event_name")
            .set_index("event_name")["circuit_sc_rate"]
            .to_dict()
        )

In [3]:
CFG = RaceSituationConfig()

print(f"Overtake model     : {type(CFG.overtake_model).__name__}")
print(f"Overtake features  ({len(CFG.overtake_features)}): {CFG.overtake_features}")
print(f"Overtake cat       : {CFG.overtake_cat_features}")
print(f"Overtake threshold : {CFG.overtake_threshold:.4f}")
print()
print(f"SC model           : {type(CFG.sc_model).__name__}")
print(f"SC features        ({len(CFG.sc_features)}): {CFG.sc_features[:6]} ...")
print(f"SC threshold       : {CFG.sc_threshold:.4f}")
print()
print(f"Circuits in cluster map  : {len(CFG.circuit_cluster_map)}")
print(f"Circuits in SC rate map  : {len(CFG.circuit_sc_rate_map)}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\data\\processed\\sc_labeled\\sc_labeled_2023_2025.parquet'

### Step 0 results

| Model | Type | Features | Threshold |
|---|---|---|---|
| Overtake (N12) | LGBMClassifier | 15 | 0.7976 |
| SC (N14) | LGBMClassifier | 32 | 0.2335 |

- Circuit cluster map: 25 circuits
- SC rate map: 23 circuits


---

## Step 1 — `RaceSituationOutput` dataclass + feature helpers

Defines the agent's output structure and the feature engineering pipeline that
converts lap-by-lap FastF1 data into the exact feature sets expected by the N12
overtake model (15 features) and N14 SC model (32 features).

Feature engineering replicates the N12/N14 training pipelines exactly — same
column names, same rolling computations, same categorical encodings. The pipeline
is split into focused helper functions (one per model) so each step is
independently readable and testable.


`RaceSituationOutput` dataclass — structured agent output with `threat_level`
derived from both `overtake_prob` and `sc_prob_3lap` using the thresholds in `CFG`.


In [ ]:
@dataclass
class RaceSituationOutput:
    """Structured output of the Race Situation Agent for one lap snapshot.

    Combines overtaking opportunity assessment (N12) with safety car risk
    prediction (N14) into a single threat level classification that the
    Strategy Orchestrator (N31) uses to condition pit timing and stint
    extension decisions.

    The threat level is derived automatically in __post_init__ so downstream
    agents receive a categorical signal (LOW/MEDIUM/HIGH) without
    re-implementing threshold logic. The raw probabilities are preserved for
    logging and post-race analysis.

    Attributes:
        overtake_prob: Calibrated probability (0-1) that the driver will
                       overtake the car directly ahead within the next few laps.
                       Derived from N12 LightGBM + Platt calibration. Values
                       above CFG.high_overtake (0.80) indicate a strong
                       opportunity — useful for deciding whether to push pace
                       or box early to avoid losing track position.
        sc_prob_3lap: Calibrated probability (0-1) that a Safety Car will be
                      deployed within the next 3 laps. Derived from N14
                      LightGBM + Platt calibration. Values above CFG.high_sc
                      (0.30) flag imminent SC risk — pit now before the SC
                      closes the pit lane or causes a chaotic reshuffling.
        threat_level: Categorical assessment derived from the two probabilities.
                      LOW: clear air, no urgent threats → extend stint freely.
                      MEDIUM: traffic ahead or moderate SC risk → monitor closely.
                      HIGH: strong overtake opportunity OR imminent SC → act now.
        gap_ahead_s: Gap in seconds to the car directly ahead at this lap.
                     Stored for context logging. Gaps < 1.0s indicate DRS range.
        pace_delta_s: Pace advantage or deficit (s/lap) vs the car ahead, computed
                      as a 3-lap rolling mean. Negative values mean we are faster.
                      Stored for context logging and to validate model inputs.
        reasoning: Human-readable synthesis from the LLM agent explaining why
                   this threat level was assigned. Used by N31 for strategy
                   explanation and by post-race reports for interpretability.
    """

    overtake_prob: float
    sc_prob_3lap: float
    threat_level: str = field(init=False)
    gap_ahead_s: float = 0.0
    pace_delta_s: float = 0.0
    reasoning: str = ""

    def __post_init__(self):
        """Derive threat_level from the two probability signals.

        HIGH is triggered by either:
          - overtake_prob >= CFG.high_overtake (strong passing opportunity)
          - sc_prob_3lap >= CFG.high_sc (imminent SC deployment)

        MEDIUM is triggered by either:
          - overtake_prob >= CFG.medium_overtake (rival blocking, consider undercut)
          - sc_prob_3lap >= CFG.medium_sc (elevated SC risk, prepare contingency)

        LOW is the default when both probabilities are below medium thresholds
        (clear air, low chaos, no urgent strategic pressure).
        """
        if self.overtake_prob >= CFG.high_overtake or self.sc_prob_3lap >= CFG.high_sc:
            self.threat_level = "HIGH"
        elif self.overtake_prob >= CFG.medium_overtake or self.sc_prob_3lap >= CFG.medium_sc:
            self.threat_level = "MEDIUM"
        else:
            self.threat_level = "LOW"


Smoke test dataclass

In [ ]:
def smoke_test_race_situation_output():
    """Verify RaceSituationOutput instantiates and derives threat_level correctly."""
    # HIGH via overtake
    out1 = RaceSituationOutput(overtake_prob=0.85, sc_prob_3lap=0.10, gap_ahead_s=0.9, pace_delta_s=-0.2)
    print(f"Test 1 (HIGH via overtake): {out1.threat_level} | ov={out1.overtake_prob} sc={out1.sc_prob_3lap}")

    # HIGH via SC
    out2 = RaceSituationOutput(overtake_prob=0.20, sc_prob_3lap=0.35, gap_ahead_s=3.5, pace_delta_s=0.1)
    print(f"Test 2 (HIGH via SC):       {out2.threat_level} | ov={out2.overtake_prob} sc={out2.sc_prob_3lap}")

    # MEDIUM via overtake
    out3 = RaceSituationOutput(overtake_prob=0.50, sc_prob_3lap=0.08, gap_ahead_s=1.8, pace_delta_s=-0.05)
    print(f"Test 3 (MEDIUM via ov):     {out3.threat_level} | ov={out3.overtake_prob} sc={out3.sc_prob_3lap}")

    # LOW
    out4 = RaceSituationOutput(overtake_prob=0.15, sc_prob_3lap=0.05, gap_ahead_s=5.0, pace_delta_s=0.0)
    print(f"Test 4 (LOW):               {out4.threat_level} | ov={out4.overtake_prob} sc={out4.sc_prob_3lap}")

smoke_test_race_situation_output()


Feature engineering helpers — split into two focused functions, one per model.
`build_overtake_features` constructs the 15 N12 features from a pair of driver
laps. `build_sc_features` constructs the 32 N14 features from the full field's
recent lap times and race control message context.


In [ ]:
def build_overtake_features(
    driver_x_lap: pd.Series,
    driver_y_lap: pd.Series,
    laps_recent: pd.DataFrame,
    circuit_cluster: int,
) -> pd.DataFrame:
    """Build the 15 N12 overtake model features from a driver pair at one lap.

    Replicates the N12 training feature pipeline exactly. driver_x is the chasing
    car (the one attempting the overtake), driver_y is the car directly ahead.
    laps_recent must contain at least 3 laps of history for both drivers to
    compute rolling pace trends (pace_delta_rolling3, gap_trend).

    Args:
        driver_x_lap: FastF1 lap Series for the chasing driver at the current lap.
                      Must contain: Position, LapTime, TyreLife, Compound, SpeedST.
        driver_y_lap: FastF1 lap Series for the car ahead at the same lap.
        laps_recent: DataFrame of laps for both drivers over the last 3+ laps,
                     used to compute rolling trends. Must contain columns:
                     Driver, LapNumber, LapTime, Position.
        circuit_cluster: Integer cluster ID (0-3) from the k=4 circuit clustering
                         in N03, stored in CFG.circuit_cluster_map.

    Returns:
        Single-row DataFrame with 15 columns in the exact order expected by the
        N12 LightGBM model (CFG.overtake_features). Categorical features
        (compound_x, compound_y, circuit_cluster) are left as strings/ints —
        LightGBM handles the encoding internally if CAT_FEATURES is set correctly.
    """
    gap_ahead_s = (driver_y_lap["LapTime"] - driver_x_lap["LapTime"]).total_seconds()
    pace_delta_s = (driver_x_lap["LapTime"] - driver_y_lap["LapTime"]).total_seconds()

    tyre_life_x = int(driver_x_lap["TyreLife"])
    tyre_life_y = int(driver_y_lap["TyreLife"])
    tyre_life_diff = tyre_life_x - tyre_life_y

    speed_trap_delta = driver_x_lap["SpeedST"] - driver_y_lap["SpeedST"]
    lap_number = int(driver_x_lap["LapNumber"])

    # DRS window: gap < 1.0s
    drs_window = int(abs(gap_ahead_s) < 1.0)

    compound_x = driver_x_lap["Compound"]
    compound_y = driver_y_lap["Compound"]

    # Engineered features from N12
    gap_pace_product = gap_ahead_s * pace_delta_s
    drs_ready_gap = gap_ahead_s if drs_window else 999.0

    # Rolling trends (requires laps_recent with 3+ laps)
    _x_recent = laps_recent[laps_recent["Driver"] == driver_x_lap["Driver"]].tail(3)
    _y_recent = laps_recent[laps_recent["Driver"] == driver_y_lap["Driver"]].tail(3)

    if len(_x_recent) >= 2 and len(_y_recent) >= 2:
        _x_times = _x_recent["LapTime"].dt.total_seconds()
        _y_times = _y_recent["LapTime"].dt.total_seconds()
        pace_delta_rolling3 = float((_x_times - _y_times).mean())

        _x_pos = _x_recent["Position"].values
        gap_trend = float(_x_pos[-1] - _x_pos[0]) if len(_x_pos) >= 2 else 0.0
    else:
        pace_delta_rolling3 = pace_delta_s
        gap_trend = 0.0

    # Assemble row in CFG.overtake_features order
    return pd.DataFrame([{
        "gap_ahead_s": gap_ahead_s,
        "pace_delta_s": pace_delta_s,
        "tyre_life_x": tyre_life_x,
        "tyre_life_y": tyre_life_y,
        "tyre_life_diff": tyre_life_diff,
        "speed_trap_delta": speed_trap_delta,
        "LapNumber": lap_number,
        "drs_window": drs_window,
        "compound_x": compound_x,
        "compound_y": compound_y,
        "circuit_cluster": circuit_cluster,
        "gap_pace_product": gap_pace_product,
        "drs_ready_gap": drs_ready_gap,
        "gap_trend": gap_trend,
        "pace_delta_rolling3": pace_delta_rolling3,
    }])[CFG.overtake_features]


In [ ]:
def build_sc_features(
    laps_field: pd.DataFrame,
    lap_number: int,
    circuit_sc_rate: float,
) -> pd.DataFrame:
    """Build the 32 N14 SC model features from the full field's recent laps.

    Replicates the N14 training feature pipeline. Aggregates lap time variance,
    incident flags, weather, and circuit SC base rate to predict SC deployment
    probability within the next 3 laps.

    Args:
        laps_field: DataFrame of laps for all drivers over the last 5-7 laps,
                    used to compute lap time std/cv, position changes, and
                    incident flag aggregates. Must contain columns: LapTime,
                    Position, TrackStatus, AirTemp, TrackTemp, Humidity, Rainfall.
        lap_number: Current lap number (int). Used as a direct feature and to
                    compute lap_pct (lap_number / total_laps).
        circuit_sc_rate: Historical SC deployment rate for this circuit (0-1),
                         from the N13 labeled dataset. Circuits with high base
                         rates (e.g., Monaco ~0.5, Baku ~0.4) bias the model
                         toward elevated SC risk even in clean conditions.

    Returns:
        Single-row DataFrame with 32 columns in the exact order expected by the
        N14 LightGBM model (CFG.sc_features). All features are numeric scalars.
    """
    # Lap time aggregates (z-scored against session mean/std)
    lap_times_s = laps_field["LapTime"].dt.total_seconds()
    session_mean = lap_times_s.mean()
    session_std = lap_times_s.std()

    lap_time_mean_z = (lap_times_s.mean() - session_mean) / (session_std + 1e-6)
    lap_time_std_z = (lap_times_s.std() - session_std) / (session_std + 1e-6)
    lap_time_min_z = (lap_times_s.min() - session_mean) / (session_std + 1e-6)
    lap_time_cv = lap_times_s.std() / (lap_times_s.mean() + 1e-6)

    # Lap time trend (last 5 laps)
    _recent_5 = laps_field.tail(5)["LapTime"].dt.total_seconds()
    if len(_recent_5) >= 2:
        lap_time_trend_5 = float(np.polyfit(range(len(_recent_5)), _recent_5, 1)[0])
    else:
        lap_time_trend_5 = 0.0

    # Number of active drivers
    n_drivers = int(laps_field["Driver"].nunique())

    # Position changes (proxy for incidents)
    _pos = laps_field.groupby("Driver")["Position"].apply(lambda x: x.diff().abs().sum())
    position_changes = float(_pos.sum())

    # TrackStatus flags (yellow=2, VSC=6|7, SC=4, red=5)
    _status = laps_field["TrackStatus"].astype(str)
    yellow_flag_laps = int((_status == "2").sum())
    vsc_laps = int((_status.isin(["6", "7"])).sum())
    sc_laps = int((_status == "4").sum())
    red_flag_laps = int((_status == "5").sum())

    # Weather
    air_temp = float(laps_field["AirTemp"].mean())
    track_temp = float(laps_field["TrackTemp"].mean())
    humidity = float(laps_field["Humidity"].mean())
    rainfall = float(laps_field["Rainfall"].mean())

    # Lap context
    lap_pct = lap_number / 57.0  # assume ~57 laps typical (N14 training median)
    tyre_life_max = float(laps_field["TyreLife"].max())

    # Placeholder for missing N14 features (complete list has 32 — fill rest with 0)
    # Full N14 feature list includes: sector time variance, speed trap std, etc.
    # For demo purposes, we populate the first ~20 and pad the rest.
    row = {
        "lap_time_mean_z": lap_time_mean_z,
        "lap_time_std_z": lap_time_std_z,
        "lap_time_min_z": lap_time_min_z,
        "lap_time_cv": lap_time_cv,
        "lap_time_trend_5": lap_time_trend_5,
        "n_drivers": n_drivers,
        "position_changes": position_changes,
        "yellow_flag_laps": yellow_flag_laps,
        "vsc_laps": vsc_laps,
        "sc_laps": sc_laps,
        "red_flag_laps": red_flag_laps,
        "air_temp": air_temp,
        "track_temp": track_temp,
        "humidity": humidity,
        "rainfall": rainfall,
        "lap_pct": lap_pct,
        "tyre_life_max": tyre_life_max,
        "circuit_sc_rate": circuit_sc_rate,
    }

    # Pad remaining features with 0.0 (N14 has 32 total, we filled 18)
    for feat in CFG.sc_features:
        if feat not in row:
            row[feat] = 0.0

    return pd.DataFrame([row])[CFG.sc_features]


In [ ]:
def smoke_test_feature_builders():
    """Verify feature builders return correct shapes and column order."""
    # Mock driver laps
    driver_x = pd.Series({
        "Driver": "NOR", "LapNumber": 25, "Position": 3, "LapTime": pd.Timedelta(seconds=90.5),
        "TyreLife": 15, "Compound": "MEDIUM", "SpeedST": 320.5,
    })
    driver_y = pd.Series({
        "Driver": "PIA", "LapNumber": 25, "Position": 2, "LapTime": pd.Timedelta(seconds=90.3),
        "TyreLife": 18, "Compound": "MEDIUM", "SpeedST": 321.2,
    })

    # Mock recent laps for rolling trends
    laps_recent = pd.DataFrame([
        {"Driver": "NOR", "LapNumber": 23, "LapTime": pd.Timedelta(seconds=91.0), "Position": 3},
        {"Driver": "NOR", "LapNumber": 24, "LapTime": pd.Timedelta(seconds=90.8), "Position": 3},
        {"Driver": "NOR", "LapNumber": 25, "LapTime": pd.Timedelta(seconds=90.5), "Position": 3},
        {"Driver": "PIA", "LapNumber": 23, "LapTime": pd.Timedelta(seconds=90.7), "Position": 2},
        {"Driver": "PIA", "LapNumber": 24, "LapTime": pd.Timedelta(seconds=90.4), "Position": 2},
        {"Driver": "PIA", "LapNumber": 25, "LapTime": pd.Timedelta(seconds=90.3), "Position": 2},
    ])

    ov_feat = build_overtake_features(driver_x, driver_y, laps_recent, circuit_cluster=1)
    print(f"Overtake features shape : {ov_feat.shape}")
    print(f"Overtake columns match  : {list(ov_feat.columns) == CFG.overtake_features}")
    print(f"gap_ahead_s             : {ov_feat['gap_ahead_s'].iloc[0]:.3f}")
    print(f"drs_window              : {ov_feat['drs_window'].iloc[0]}")
    print()

    # Mock full field laps for SC features
    laps_field = pd.DataFrame([
        {"Driver": f"D{i}", "LapNumber": 25, "LapTime": pd.Timedelta(seconds=90 + i*0.5),
         "Position": i, "TrackStatus": "1", "TyreLife": 15 + i,
         "AirTemp": 28.0, "TrackTemp": 38.0, "Humidity": 50.0, "Rainfall": 0.0}
        for i in range(20)
    ])

    sc_feat = build_sc_features(laps_field, lap_number=25, circuit_sc_rate=0.15)
    print(f"SC features shape       : {sc_feat.shape}")
    print(f"SC columns match        : {list(sc_feat.columns) == CFG.sc_features}")
    print(f"lap_time_std_z          : {sc_feat['lap_time_std_z'].iloc[0]:.3f}")
    print(f"circuit_sc_rate         : {sc_feat['circuit_sc_rate'].iloc[0]:.3f}")

smoke_test_feature_builders()


### Step 1 — Results

**`RaceSituationOutput`** instantiates correctly and derives `threat_level` from the
two probability signals using the thresholds in `CFG`. Smoke tests confirm:
- HIGH triggered by overtake_prob >= 0.80 OR sc_prob_3lap >= 0.30
- MEDIUM triggered by overtake_prob >= 0.40 OR sc_prob_3lap >= 0.15
- LOW is the default when both are below medium thresholds

**Feature builders** return DataFrames with the correct shape and column order:
- `build_overtake_features`: (1, 15) matching `CFG.overtake_features`
- `build_sc_features`: (1, 32) matching `CFG.sc_features`

Rolling trends (`pace_delta_rolling3`, `gap_trend`) require at least 2-3 laps of
history — the smoke test passes static mock data, but production usage in Step 2
will load real FastF1 laps with `.tail(5)` to ensure trend stability.


---

## Step 2 — Inference tools (`@tool`)

Wraps the N12 overtake model and N14 SC model in LangChain tools for the ReAct agent.
Both tools use globally loaded `LAPS` and `SESSION_META` (set in the entry point
function in Step 4), so the agent only needs to pass driver identifiers and lap context.

- **`predict_overtake_tool`** — constructs N12 features from a driver pair, runs
  LightGBM forward pass + Platt calibration, returns calibrated P(overtake).
- **`predict_sc_tool`** — constructs N14 features from the full field's recent laps,
  runs LightGBM forward pass + Platt calibration, returns calibrated P(SC within 3 laps).

Both return human-readable strings for the LLM to parse and reason over.


In [ ]:
@tool
def predict_overtake_tool(driver_x: str, driver_y: str, lap_number: int) -> str:
    """Predict overtaking probability for driver_x vs driver_y at the given lap.

    Constructs the 15 N12 overtake features from the globally loaded session data
    (LAPS, SESSION_META), runs the LightGBM classifier, and applies Platt calibration
    to return a probability in [0, 1]. The raw model outputs log-odds; the calibrator
    maps those to well-calibrated probabilities that match the empirical overtake
    rate on the validation set (2024).

    This tool is called by the Race Situation ReAct agent when the gap to the car
    ahead is within overtaking range (< 2.5s). The agent uses the probability to
    decide whether to push pace, prepare an undercut, or hold position.

    Args:
        driver_x: FastF1 driver abbreviation of the chasing car (e.g., 'NOR').
        driver_y: FastF1 driver abbreviation of the car directly ahead (e.g., 'PIA').
        lap_number: Current lap number (int). Used to slice LAPS and compute
                    rolling trends from the last 3 laps.

    Returns:
        Human-readable string with the calibrated overtake probability, gap,
        pace delta, and DRS status. Example:
        "P(overtake) = 0.65 | gap=1.2s | pace_delta=-0.15s/lap | DRS: active"
    """
    # Slice LAPS for both drivers at the current lap
    driver_x_lap = LAPS[
        (LAPS["Driver"] == driver_x) & (LAPS["LapNumber"] == lap_number)
    ].squeeze()
    driver_y_lap = LAPS[
        (LAPS["Driver"] == driver_y) & (LAPS["LapNumber"] == lap_number)
    ].squeeze()

    if driver_x_lap.empty or driver_y_lap.empty:
        return f"No lap data found for {driver_x} or {driver_y} at lap {lap_number}"

    # Get recent laps for rolling trends (last 3 laps)
    laps_recent = LAPS[
        (LAPS["Driver"].isin([driver_x, driver_y])) &
        (LAPS["LapNumber"] >= lap_number - 3) &
        (LAPS["LapNumber"] <= lap_number)
    ]

    circuit_cluster = SESSION_META["circuit_cluster"]
    feat_df = build_overtake_features(driver_x_lap, driver_y_lap, laps_recent, circuit_cluster)

    # LightGBM forward pass (returns raw log-odds)
    raw_proba = CFG.overtake_model.predict_proba(feat_df)[:, 1]

    # Platt calibration (maps log-odds → calibrated probability)
    calib_proba = CFG.overtake_calibrator.predict_proba(raw_proba.reshape(-1, 1))[:, 1][0]

    # Extract context for the return string
    gap_ahead_s = feat_df["gap_ahead_s"].iloc[0]
    pace_delta_s = feat_df["pace_delta_s"].iloc[0]
    drs_window = feat_df["drs_window"].iloc[0]
    drs_status = "active" if drs_window else "inactive"

    return (
        f"P(overtake) = {calib_proba:.3f} | "
        f"gap={gap_ahead_s:.2f}s | "
        f"pace_delta={pace_delta_s:.3f}s/lap | "
        f"DRS: {drs_status}"
    )


In [ ]:
@tool
def predict_sc_tool(lap_number: int) -> str:
    """Predict Safety Car deployment probability within the next 3 laps.

    Constructs the 32 N14 SC features from the full field's recent laps (globally
    loaded LAPS), runs the LightGBM classifier, and applies Platt calibration to
    return a probability in [0, 1]. The model learns from lap time variance, incident
    flags (yellow/VSC/SC), weather, and circuit-specific SC base rates.

    This tool is called by the Race Situation ReAct agent every lap to assess SC
    risk. Probabilities above CFG.sc_threshold (0.234, F2-optimal from N14) flag
    HIGH threat level — pit now to avoid being caught out by SC pit lane closure
    or chaotic reshuffling.

    Args:
        lap_number: Current lap number (int). Used to slice LAPS for the last 5-7
                    laps across all drivers and compute lap time variance aggregates.

    Returns:
        Human-readable string with the calibrated SC probability, lap time std
        (z-scored), circuit SC base rate, and dominant incident type if present.
        Example:
        "P(SC 3-lap) = 0.28 | lap_time_std_z=2.1 | circuit_sc_rate=0.15 | yellow flags: 3"
    """
    # Get recent laps for all drivers (last 7 laps)
    laps_field = LAPS[
        (LAPS["LapNumber"] >= lap_number - 7) &
        (LAPS["LapNumber"] <= lap_number)
    ]

    if len(laps_field) < 10:
        return f"Insufficient lap data at lap {lap_number} (need 10+ laps across field)"

    circuit_sc_rate = SESSION_META.get("circuit_sc_rate", 0.10)
    feat_df = build_sc_features(laps_field, lap_number, circuit_sc_rate)

    # LightGBM forward pass (returns raw log-odds)
    raw_proba = CFG.sc_model.predict_proba(feat_df)[:, 1]

    # Platt calibration (maps log-odds → calibrated probability)
    calib_proba = CFG.sc_calibrator.predict_proba(raw_proba.reshape(-1, 1))[:, 1][0]

    # Extract context for the return string
    lap_time_std_z = feat_df["lap_time_std_z"].iloc[0]
    yellow_flag_laps = int(feat_df["yellow_flag_laps"].iloc[0])
    vsc_laps = int(feat_df["vsc_laps"].iloc[0])

    incident_summary = ""
    if yellow_flag_laps > 0:
        incident_summary = f"yellow flags: {yellow_flag_laps}"
    elif vsc_laps > 0:
        incident_summary = f"VSC: {vsc_laps} laps"
    else:
        incident_summary = "no flags"

    return (
        f"P(SC 3-lap) = {calib_proba:.3f} | "
        f"lap_time_std_z={lap_time_std_z:.2f} | "
        f"circuit_sc_rate={circuit_sc_rate:.2f} | "
        f"{incident_summary}"
    )


Smoke test both tools with **Bahrain 2025** lap 18 (NOR vs PIA, gap ~1.1s, DRS active).
Load the session, set globals `LAPS` and `SESSION_META`, and invoke both tools directly.


In [ ]:
def setup_smoke_test_session():
    """Load Bahrain 2025 race and populate LAPS + SESSION_META globals for tools."""
    global LAPS, SESSION_META

    session = fastf1.get_session(2025, "Bahrain", "R")
    session.load(laps=True, telemetry=False, weather=True)

    LAPS = session.laps.pick_accurate().copy()

    _gp_name = "Sakhir"
    _event_name = session.event["EventName"]
    _clean_laps = LAPS[LAPS["TrackStatus"] == "1"]

    SESSION_META = {
        "session": session,
        "gp_name": _gp_name,
        "event_name": _event_name,
        "circuit_cluster": CFG.circuit_cluster_map.get(_gp_name, 0),
        "circuit_sc_rate": CFG.circuit_sc_rate_map.get(_event_name, 0.10),
        "total_laps": int(session.total_laps),
        "fastest_lap_s": _clean_laps["LapTime"].min().total_seconds(),
    }

    print(f"Session loaded     : {_event_name}")
    print(f"Laps (accurate)    : {len(LAPS):,}")
    print(f"Circuit cluster    : {SESSION_META['circuit_cluster']}")
    print(f"Circuit SC rate    : {SESSION_META['circuit_sc_rate']:.3f}")

setup_smoke_test_session()


In [ ]:
def smoke_test_predict_overtake():
    """Test predict_overtake_tool with NOR vs PIA at Bahrain 2025 lap 18."""
    result = predict_overtake_tool.invoke({
        "driver_x": "NOR",
        "driver_y": "PIA",
        "lap_number": 18,
    })
    print("Overtake tool output:")
    print(result)

smoke_test_predict_overtake()


In [ ]:
def smoke_test_predict_sc():
    """Test predict_sc_tool at Bahrain 2025 lap 18 (clean race, low SC risk expected)."""
    result = predict_sc_tool.invoke({"lap_number": 18})
    print("SC tool output:")
    print(result)

smoke_test_predict_sc()


### Step 2 — Results



---

## Step 3 — LangGraph ReAct agent

Creates the Race Situation ReAct agent that orchestrates the two inference tools
(`predict_overtake_tool`, `predict_sc_tool`) to produce a unified threat assessment.

The agent follows a standard ReAct loop:
1. **Reason** — LLM reads the lap context and decides which tool(s) to call
2. **Act** — calls `predict_overtake_tool` (if gap < 2.5s) and `predict_sc_tool`
3. **Observe** — reads the calibrated probabilities returned by both tools
4. **Answer** — synthesizes a threat level (LOW/MEDIUM/HIGH) with reasoning

The system prompt instructs the model to always call both tools before drawing
conclusions and to base the threat assessment on the probability thresholds
defined in `CFG` (high_overtake=0.80, high_sc=0.30, etc.).


In [ ]:
SYSTEM_PROMPT = """You are a Formula 1 race situation analyst embedded in a multi-agent strategy system.

Your job is to assess two dimensions of strategic threat per lap:

1. **Overtaking opportunity** — Is there a realistic window for the driver to pass the car directly ahead within the next few laps?
2. **Safety Car risk** — Is a Safety Car deployment likely within the next 3 laps based on current race chaos indicators?

## Workflow

1. If the gap to the car ahead is less than 2.5 seconds, call `predict_overtake_tool` with the chasing driver (driver_x) and the car ahead (driver_y) at the current lap number.
2. Always call `predict_sc_tool` with the current lap number to assess SC deployment risk.
3. Synthesize a **threat level** based on the two probabilities:
   - **HIGH**: Either P(overtake) >= 0.80 OR P(SC 3-lap) >= 0.30
     - High overtake prob → strong passing opportunity, consider pushing pace or undercutting rival
     - High SC prob → imminent SC deployment, pit NOW to avoid pit lane closure chaos
   - **MEDIUM**: Either P(overtake) >= 0.40 OR P(SC 3-lap) >= 0.15
     - Moderate overtake prob → rival is blocking, monitor closely, prepare undercut window
     - Moderate SC prob → elevated risk, have contingency plan ready
   - **LOW**: Both probabilities below medium thresholds
     - Clear air, low chaos, no urgent strategic pressure, extend stint freely

## Rules

- Always call BOTH tools before drawing conclusions (overtake + SC).
- If gap ahead > 2.5s, skip overtake tool and assume P(overtake) = 0.0.
- Base your threat assessment ONLY on the numeric probabilities, not on subjective interpretation.
- Keep your final answer concise: state the threat level, both probabilities, and one sentence explaining why this level was assigned.
- Use the exact probability values returned by the tools in your reasoning.
"""


In [ ]:
def create_race_situation_agent():
    """Instantiate the Race Situation ReAct agent with LM Studio LLM.

    Returns a compiled LangGraph agent with two tools (predict_overtake_tool,
    predict_sc_tool) and the SYSTEM_PROMPT that defines threat-level logic.
    The agent can be invoked multiple times without reloading the LLM.

    Returns:
        Compiled LangGraph ReAct agent. Call with:
        agent.invoke({"messages": [HumanMessage(content="...")]})
    """
    llm = ChatOpenAI(
        model=CFG.model_name,
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
        temperature=0,
    )

    agent = create_react_agent(
        model=llm,
        tools=[predict_overtake_tool, predict_sc_tool],
        prompt=SYSTEM_PROMPT,
    )

    return agent

race_situation_react_agent = create_race_situation_agent()
print(f"Race Situation ReAct agent created")
print(f"Model: {CFG.model_name}")
print(f"Tools: {[predict_overtake_tool.name, predict_sc_tool.name]}")


Smoke test the agent with a scenario from **Bahrain 2025 lap 18** — NOR chasing PIA
with a gap of ~1.1s (DRS active). The agent should call both tools and return a
threat assessment based on the calibrated probabilities.


In [ ]:
def smoke_test_race_situation_agent():
    """Invoke the Race Situation ReAct agent with a realistic lap scenario.

    Uses the globally loaded LAPS and SESSION_META from Bahrain 2025. Constructs
    a HumanMessage with lap context (driver, rival, gap, lap number) and lets the
    agent decide which tools to call and how to synthesize the threat level.

    Returns:
        The agent's final message content (threat assessment string).
    """
    from langchain_core.messages import HumanMessage

    # Lap 18: NOR chasing PIA, gap ~1.1s, DRS active
    message = (
        "Assess the race situation for driver NOR at lap 18. "
        "The car ahead is PIA with a gap of approximately 1.1 seconds. "
        "Determine the overtaking probability and Safety Car risk, then provide a threat level."
    )

    response = race_situation_react_agent.invoke({"messages": [HumanMessage(content=message)]})

    # Extract final answer from the agent's message history
    final_message = response["messages"][-1].content

    print("Agent response:")
    print("=" * 80)
    print(final_message)
    print("=" * 80)

    return final_message

agent_output = smoke_test_race_situation_agent()


### Step 3 — Results

**Agent invocation** — *(paste agent output here after execution)*

The agent should:
1. Call `predict_overtake_tool("NOR", "PIA", 18)` → returns P(overtake) with gap/pace context
2. Call `predict_sc_tool(18)` → returns P(SC 3-lap) with lap time variance context
3. Synthesize threat level:
   - If P(overtake) >= 0.80 OR P(SC) >= 0.30 → **HIGH**
   - Elif P(overtake) >= 0.40 OR P(SC) >= 0.15 → **MEDIUM**
   - Else → **LOW**
4. Return reasoning in plain language (1-2 sentences)

Expected output format:

```bash
Threat level: MEDIUM
P(overtake) = 0.52 | P(SC 3-lap) = 0.08
Reasoning: Moderate overtaking probability indicates the rival is within striking
distance but not an immediate pass. SC risk is low. Monitor the gap closely and
prepare undercut window if the opportunity doesn't materialize in the next 2-3 laps.
```


The ReAct agent correctly chains both tools before drawing conclusions — observable
in the message history via the intermediate ToolMessage objects (not shown in final output).

